# Modeling

Training and comparing Logistic Regression, Decision Tree, and CatBoost.

In [7]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import load_data
from src.models import train_logistic_regression, train_decision_tree, train_catboost
from src.evaluation import evaluate_model, cross_validate_model

In [8]:
df = load_data()

X = df.drop(columns=["y"])
y = (df["y"] == "yes").astype(int)

categorical_features = X.select_dtypes(include=["string"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
model_lr = train_logistic_regression(X_train, y_train, categorical_features, numerical_features)
model_dt = train_decision_tree(X_train, y_train, categorical_features, numerical_features)
model_cb = train_catboost(X_train, y_train, categorical_features)

In [10]:
results = [
  evaluate_model(model_lr, X_test, y_test, "Logistic Regression"),
  evaluate_model(model_dt, X_test, y_test, "Decision Tree"),
  evaluate_model(model_cb, X_test, y_test, "CatBoost")
]

pd.DataFrame(results)

,Model,Accuracy,F1,ROC-AUC
0,Logistic Regression,0.8933,0.2815,0.7717
1,Decision Tree,0.8919,0.2892,0.7136
2,CatBoost,0.8974,0.3911,0.8034


In [11]:
cv_results = [
  cross_validate_model(train_logistic_regression, X, y, "Logistic Regression", 
             categorical_features=categorical_features, numerical_features=numerical_features),
  cross_validate_model(train_decision_tree, X, y, "Decision Tree", 
             categorical_features=categorical_features, numerical_features=numerical_features),
  cross_validate_model(train_catboost, X, y, "CatBoost", 
             categorical_features=categorical_features)
]

pd.DataFrame(cv_results)

,Model,Mean ROC-AUC,Std
0,Logistic Regression,0.7654,0.0084
1,Decision Tree,0.7082,0.0070
2,CatBoost,0.8025,0.0093


In [12]:
feature_importance = pd.DataFrame({
  "Feature": X.columns,
  "Importance": model_cb.get_feature_importance()
}).sort_values("Importance", ascending=False)

feature_importance.head(10)

,Feature,Importance
10,month,17.998935
9,day,13.998923
14,poutcome,9.264185
8,contact,9.170703
5,balance,9.153548
0,age,8.616625
12,pdays,6.574766
6,housing,4.979680
1,job,4.913559
11,campaign,4.203459
